# UGC Comment Region Detection — scikit-learn Notebook

This notebook lets you pull your live labeled dataset from the research server, explore it, and train scikit-learn models exactly the same way the production system does — so you can experiment locally or on Google Colab without touching the server.

**What you get:**
- ~13,000 labeled candidate rows, each with 81 extracted features
- Binary label: `1` = genuine UGC comment region, `0` = not a comment region
- One row per candidate DOM element from a real crawled page

---

## 0 — Install dependencies

Run this once. On Colab everything except `requests` is already installed.

In [ ]:
!pip install -q pandas scikit-learn matplotlib seaborn requests

## 1 — Configuration

Set your server URL and the job ID you want to work with.  
The **Unified Dataset** job is the right one — it aggregates all your labeled pages.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
API_BASE  = 'https://api.xsscommentdetection.me'

# The Unified Dataset job — aggregates all labeled pages
JOB_ID    = '1e6a5b03-ee5e-46e4-904a-838f990f8715'

# Model variant — 'keyword-aware' includes lexical keyword features
#                 'keyword-ablated' drops them (structural signals only)
VARIANT   = 'keyword-aware'

# Auth token — get this from Admin → Shareable Access Links → generate supervisor link
# then sign in and copy your token from localStorage in the browser console:
#   localStorage.getItem('ugc_auth_token')
AUTH_TOKEN = ''   # leave empty if the endpoint doesn't require auth (currently open)

## 2 — Download the dataset

The server exports a CSV with every labeled candidate, all feature values, and the binary label.  
This may take 10–30 seconds depending on your connection.

In [ ]:
import requests
import pandas as pd
from io import StringIO

url = f'{API_BASE}/api/modeling/dataset.csv?variantId={VARIANT}&jobIds={JOB_ID}'
headers = {'Authorization': f'Bearer {AUTH_TOKEN}'} if AUTH_TOKEN else {}

print('Downloading dataset...')
resp = requests.get(url, headers=headers, timeout=120)
resp.raise_for_status()

df_raw = pd.read_csv(StringIO(resp.text), low_memory=False)
print(f'Downloaded: {len(df_raw):,} rows, {len(df_raw.columns)} columns')
df_raw.head(3)

## 3 — Understand what you have

In [ ]:
# Columns split into metadata, label, and feature columns
META_COLS = [
    'dataset_row_id', 'job_id', 'item_id', 'row_number', 'candidate_key',
    'candidate_rank', 'normalized_url', 'final_url', 'hostname', 'frame_url',
    'frame_host', 'analysis_source', 'manual_captured_at', 'item_status',
    'item_ugc_detected', 'candidate_screenshot_url', 'item_screenshot_url',
    'item_manual_uploaded_screenshot_url', 'human_label', 'review_complete_binary',
    'item_positive_label_count', 'item_negative_label_count', 'sample_text',
    'score', 'confidence', 'ugc_type',
]
LABEL_COL  = 'binary_label'
FEATURE_COLS = [c for c in df_raw.columns if c not in META_COLS + [LABEL_COL]]

print(f'Label column : {LABEL_COL}')
print(f'Feature count: {len(FEATURE_COLS)}')
print(f'\nFeature names:\n{FEATURE_COLS}')

In [ ]:
# Keep only rows that have a binary label (labeled candidates)
df = df_raw[df_raw[LABEL_COL].isin([0, 1])].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

print(f'Labeled rows : {len(df):,}')
print(f'Positives    : {df[LABEL_COL].sum():,}  ({df[LABEL_COL].mean()*100:.1f}%)  ← real comment regions')
print(f'Negatives    : {(df[LABEL_COL]==0).sum():,}  ({(1-df[LABEL_COL].mean())*100:.1f}%)  ← not comment regions')

## 4 — Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ── Class distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df[LABEL_COL].value_counts()
axes[0].bar(['Not comment (0)', 'Comment region (1)'], counts.values, color=['#374151', '#3b82f6'])
axes[0].set_title('Class distribution')
axes[0].set_ylabel('Candidate count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# ── Positive rate per domain (top 20) ────────────────────────────────────────
domain_stats = df.groupby('hostname')[LABEL_COL].agg(['mean', 'count']).reset_index()
domain_stats = domain_stats[domain_stats['count'] >= 10].sort_values('mean', ascending=False).head(20)
axes[1].barh(domain_stats['hostname'], domain_stats['mean'], color='#3b82f6')
axes[1].set_title('Positive rate by domain (min 10 candidates)')
axes[1].set_xlabel('Fraction labeled as comment region')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature means: positive vs negative candidates ────────────────────────────
# Shows which features differ most between the two classes

# Only use numeric features
num_feats = df[FEATURE_COLS].select_dtypes(include='number').columns.tolist()

pos_means = df[df[LABEL_COL]==1][num_feats].mean()
neg_means = df[df[LABEL_COL]==0][num_feats].mean()

# Normalise by overall std so we can compare features on different scales
std = df[num_feats].std().replace(0, 1)
effect = ((pos_means - neg_means) / std).abs().sort_values(ascending=False)

top_n = 20
top_feats = effect.head(top_n).index.tolist()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#22c55e' if (pos_means[f] > neg_means[f]) else '#ef4444' for f in top_feats]
ax.barh(top_feats[::-1], effect[top_feats[::-1]], color=colors[::-1])
ax.set_title(f'Top {top_n} most discriminative features (|mean difference| / std)\nGreen = higher in positives, Red = higher in negatives')
ax.set_xlabel('Effect size (standardised mean difference)')
plt.tight_layout()
plt.show()

## 5 — Prepare features for scikit-learn

scikit-learn needs numeric arrays. We handle:
- Categorical features (tag_name, role_attribute, etc.) → one-hot encoding
- Missing values → fill with 0
- Domain-based train/test split (same logic the server uses)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# Identify categorical vs numeric feature columns
cat_feats = df[FEATURE_COLS].select_dtypes(exclude='number').columns.tolist()
num_feats  = df[FEATURE_COLS].select_dtypes(include='number').columns.tolist()

print(f'Numeric features   : {len(num_feats)}')
print(f'Categorical features: {len(cat_feats)} — {cat_feats}')

# Build a preprocessor that one-hot encodes categoricals and passes numbers through
preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough', num_feats),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_feats),
], remainder='drop')

X = df[FEATURE_COLS].fillna(0)
y = df[LABEL_COL].values
print(f'\nX shape: {X.shape},  y shape: {y.shape}')

In [ ]:
# ── Domain-based train/test split ─────────────────────────────────────────────
# The server splits by domain hash so the model is tested on sites it has
# never seen — this is the right way to evaluate generalisation.
# We replicate that here: bucket 0 = test, buckets 1-4 = train.

import hashlib

def domain_bucket(hostname, n_buckets=5):
    """FNV-1a hash matching the server's hashString() function."""
    text = str(hostname or '')
    h = 2166136261
    for ch in text.encode('utf-8'):
        h ^= ch
        h = (h * 16777619) & 0xFFFFFFFF
    return h % n_buckets

df['_bucket'] = df['hostname'].apply(domain_bucket)

HOLDOUT_BUCKET = 0   # change to 1-4 to test on different sites
train_mask = df['_bucket'] != HOLDOUT_BUCKET
test_mask  = df['_bucket'] == HOLDOUT_BUCKET

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f'Train: {len(X_train):,} rows  ({y_train.mean()*100:.1f}% positive)')
print(f'Test : {len(X_test):,}  rows  ({y_test.mean()*100:.1f}% positive)')
print(f'Test domains: {df[test_mask]["hostname"].nunique()} unique sites')

## 6 — Train models

In [ ]:
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics         import (classification_report, confusion_matrix,
                                     roc_auc_score, precision_recall_curve, roc_curve,
                                     f1_score, precision_score, recall_score)

# ── Define models to compare ───────────────────────────────────────────────────
models = {
    'Logistic Regression': Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('pre', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))
    ]),
    'Gradient Boosting': Pipeline([
        ('pre', preprocessor),
        ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
}

results = {}

for name, pipe in models.items():
    print(f'Training {name}...', end=' ', flush=True)
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    results[name] = {
        'pipe':      pipe,
        'proba':     proba,
        'pred':      pred,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall':    recall_score(y_test, pred, zero_division=0),
        'f1':        f1_score(y_test, pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_test, proba),
    }
    r = results[name]
    print(f'F1={r["f1"]:.3f}  P={r["precision"]:.3f}  R={r["recall"]:.3f}  AUC={r["roc_auc"]:.3f}')

print('\nDone!')

## 7 — Evaluate and visualise

In [ ]:
# ── Confusion matrices side by side ───────────────────────────────────────────
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred: No', 'Pred: Yes'],
                yticklabels=['True: No', 'True: Yes'])
    ax.set_title(f'{name}\nF1={r["f1"]:.3f}  P={r["precision"]:.3f}  R={r["recall"]:.3f}')

plt.suptitle('Confusion Matrices — Test Set (Bucket 0)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Precision-Recall curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#3b82f6', '#22c55e', '#f59e0b']
for (name, r), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, r['proba'])
    axes[0].plot(rec, prec, label=f'{name} (F1={r["f1"]:.3f})', color=color)

axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── ROC curves ────────────────────────────────────────────────────────────────
for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['proba'])
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={r["roc_auc"]:.3f})', color=color)

axes[1].plot([0,1],[0,1], 'k--', alpha=0.3, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importances (Random Forest) ───────────────────────────────────────
rf_pipe = results['Random Forest']['pipe']
rf_clf  = rf_pipe.named_steps['clf']
pre     = rf_pipe.named_steps['pre']

# Reconstruct feature names after one-hot encoding
cat_names = list(pre.named_transformers_['cat'].get_feature_names_out(cat_feats))
all_names = num_feats + cat_names

importances = pd.Series(rf_clf.feature_importances_, index=all_names)
top20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#3b82f6' if i < 3 else '#1d4ed8' if i < 8 else '#374151' for i in range(20)]
ax.barh(top20.index[::-1], top20.values[::-1], color=colors_bar[::-1])
ax.set_title('Top 20 Feature Importances — Random Forest')
ax.set_xlabel('Importance (fraction of total decision-making)')
plt.tight_layout()
plt.show()

print('\nTop 10:')
for feat, imp in top20.head(10).items():
    print(f'  {feat:<45} {imp:.4f}  ({imp*100:.1f}% of total)')

## 8 — Cross-validation (domain-aware, matching the server)

This replicates the exact 5-fold CV the server runs — each fold holds out one domain bucket as the test set.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.base import clone

# Domain-aware 5-fold CV
n_folds = 5
fold_results = []

# Use Random Forest for CV (change to any model from results dict)
model_name = 'Random Forest'
pipe_template = models[model_name]

for fold in range(n_folds):
    tr_mask = df['_bucket'] != fold
    te_mask = df['_bucket'] == fold

    Xtr, ytr = X[tr_mask], y[tr_mask]
    Xte, yte = X[te_mask], y[te_mask]

    if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
        print(f'Fold {fold}: skipped (insufficient class diversity)')
        continue

    pipe = clone(pipe_template)
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    # Find best threshold by F1
    thresholds = np.arange(0.05, 0.95, 0.05)
    best_f1, best_t = 0, 0.5
    for t in thresholds:
        p = (proba >= t).astype(int)
        f = f1_score(yte, p, zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t

    best_pred = (proba >= best_t).astype(int)
    row = {
        'fold':      fold,
        'train':     len(Xtr),
        'test':      len(Xte),
        'f1_050':    f1_score(yte, pred, zero_division=0),
        'best_f1':   best_f1,
        'threshold': best_t,
        'precision': precision_score(yte, best_pred, zero_division=0),
        'recall':    recall_score(yte, best_pred, zero_division=0),
        'roc_auc':   roc_auc_score(yte, proba) if len(np.unique(yte)) > 1 else None,
    }
    fold_results.append(row)
    print(f"Fold {fold}: F1@0.5={row['f1_050']:.3f}  Best F1={row['best_f1']:.3f} (t={row['threshold']:.2f})  "
          f"P={row['precision']:.3f}  R={row['recall']:.3f}  AUC={row['roc_auc']:.3f}")

cv_df = pd.DataFrame(fold_results)
print(f"\nMean Best F1 : {cv_df['best_f1'].mean():.3f} ± {cv_df['best_f1'].std():.3f}")
print(f"Mean Precision: {cv_df['precision'].mean():.3f}")
print(f"Mean Recall  : {cv_df['recall'].mean():.3f}")

## 9 — Experiment: what happens when you drop a feature family?

Disable an entire feature family and re-run CV to measure how much it contributed.

In [ ]:
# Feature families and their columns
FAMILIES = {
    'structure':          ['tag_name','role_attribute','classes_count','data_attributes_count',
                           'aria_attributes_count','node_depth','child_tag_variety','direct_child_count'],
    'repeated_blocks':    ['repeating_group_count','child_sig_id_group_count','child_xpath_star_group_count',
                           'xpath_star_group_count','min_k_threshold_pass_3','min_k_threshold_pass_5',
                           'min_k_threshold_pass_8','min_k_threshold_pass_15','sig_id_count_weak',
                           'sig_id_count_medium','sig_id_count_strong','sibling_homogeneity_score',
                           'sig_id_recursive_nesting','reply_nesting_depth'],
    'text_body':          ['text_word_count','text_char_count','text_sentence_count','has_text_content',
                           'text_contains_links','link_density','no_text_content_in_units'],
    'lexical_keywords':   ['comment_header_with_count','attributes_contain_keywords','keyword_container_high',
                           'keyword_text_high','keyword_text_med','keyword_direct_text_high',
                           'keyword_attr_name_high','keyword_attr_value_high','keyword_unit_high_coverage',
                           'submit_button_keyword_high'],
    'interaction_controls':['reply_button_unit_coverage','reaction_coverage','edit_delete_coverage',
                            'has_nearby_textarea','aligned_with_textarea','aligned_with_content_editable',
                            'submit_button_present','collapse_expand_control','pagination_load_more_adjacent'],
    'author_time_identity':['includes_author','has_avatar','author_avatar_coverage','profile_link_coverage',
                            'author_timestamp_colocated','time_datetime_per_unit','has_relative_time'],
    'negative_controls':  ['table_row_structure','add_to_cart_present','nav_header_ancestor',
                           'high_external_link_density','external_link_density_low','star_rating_no_text',
                           'price_currency_in_unit','candidate_has_script_tag','candidate_has_mxss_sink',
                           'candidate_has_inline_event_handler','candidate_has_javascript_protocol',
                           'candidate_has_embed_sink'],
}

# Choose which family to ablate (drop) — change this to experiment
ABLATE_FAMILY = 'lexical_keywords'

drop_cols = [c for c in FAMILIES.get(ABLATE_FAMILY, []) if c in FEATURE_COLS]
keep_cols = [c for c in FEATURE_COLS if c not in drop_cols]
print(f'Dropping {len(drop_cols)} features from "{ABLATE_FAMILY}"')
print(f'Remaining features: {len(keep_cols)}')

X_abl = df[keep_cols].fillna(0)
cat_abl = [c for c in cat_feats if c in keep_cols]
num_abl = [c for c in num_feats if c in keep_cols]
pre_abl = ColumnTransformer(transformers=[
    ('num', 'passthrough', num_abl),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_abl),
], remainder='drop')

abl_scores = []
for fold in range(n_folds):
    tr_mask = df['_bucket'] != fold
    te_mask = df['_bucket'] == fold
    Xtr, ytr = X_abl[tr_mask], y[tr_mask]
    Xte, yte = X_abl[te_mask], y[te_mask]
    if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2: continue
    pipe = Pipeline([('pre', clone(pre_abl)), ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))])
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    thresholds = np.arange(0.05, 0.95, 0.05)
    best_f1 = max(f1_score(yte, (proba>=t).astype(int), zero_division=0) for t in thresholds)
    abl_scores.append(best_f1)

full_scores = cv_df['best_f1'].tolist()
print(f'\nWith all features        : Mean Best F1 = {np.mean(full_scores):.3f}')
print(f'Without {ABLATE_FAMILY:<20}: Mean Best F1 = {np.mean(abl_scores):.3f}')
print(f'Delta: {np.mean(abl_scores) - np.mean(full_scores):+.3f}')

## 10 — Examine false positives

Look at candidates the model got wrong — what do they look like?

In [ ]:
# Use the best model's predictions on fold 0
best_model = results['Random Forest']['pipe']
test_df = df[test_mask].copy()
test_df['prob']      = results['Random Forest']['proba']
test_df['predicted'] = results['Random Forest']['pred']

false_positives = test_df[(test_df[LABEL_COL] == 0) & (test_df['predicted'] == 1)]
false_negatives = test_df[(test_df[LABEL_COL] == 1) & (test_df['predicted'] == 0)]

print(f'False positives: {len(false_positives)}  (predicted comment, actually not)')
print(f'False negatives: {len(false_negatives)}  (missed real comment regions)')

print('\n--- Top 5 false positives (highest confidence wrong calls) ---')
cols_to_show = ['hostname', 'prob', 'sample_text'] + ['repeating_group_count', 'link_density', 'edit_delete_coverage']
display_cols = [c for c in cols_to_show if c in false_positives.columns]
false_positives.sort_values('prob', ascending=False)[display_cols].head(5)

---

## Quick reference — API endpoints

| What | URL | Notes |
|------|-----|-------|
| Download full CSV | `GET /api/modeling/dataset.csv?variantId=keyword-aware&jobIds=<JOB_ID>` | All features + binary_label |
| List jobs | `GET /api/jobs?limit=50` | Find job IDs |
| List trained models | `GET /api/modeling/models` | Full evaluation metadata |
| Get model detail | `GET /api/modeling/models/<artifact_id>` | Weights, reliance, threshold curve |
| List feature catalog | `GET /api/modeling/features` | All 81 features with descriptions |
| List variants | `GET /api/modeling/variants` | keyword-aware / keyword-ablated |
| Trigger CV (returns job ID) | `POST /api/modeling/cross-validate` | Body: variantId, algorithm, jobIds, excludeFeatures |
| Poll CV status | `GET /api/modeling/diag-status/<job_id>` | status: running / done / error |
| Domain bucket analysis | `POST /api/modeling/domain-buckets` | Which sites fall in which fold |
| Fold investigation | `POST /api/modeling/fold-investigation` | Feature distribution shift per fold |

**Unified Dataset Job ID:** `1e6a5b03-ee5e-46e4-904a-838f990f8715`

**Server:** `https://api.xsscommentdetection.me`
